In [1]:
from langchain_core.documents import Document
from langchain_community.document_loaders.text import TextLoader

C:\Users\kkris\AppData\Local\Temp\ipykernel_14980\1826532694.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader
C:\Users\kkris\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = TextLoader("data/history.txt",encoding="utf-8")

In [3]:
#pdf data
from langchain_community.document_loaders.pdf import PyPDFLoader
pdf_loader = PyPDFLoader("data/history.pdf")

In [4]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path,filename)
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            all_docs.extend(doc)
            num_docs += 1
    print("total pdfs:", num_docs)
    print("total pages:",len(all_docs))
    return all_docs

all_pdf_documents = load_all_pdfs()
type(all_pdf_documents[1])

total pdfs: 1
total pages: 117


langchain_core.documents.base.Document

In [5]:
#chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_docs(documents, chunk_size = 500, chunk_overlap = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

chunks = split_docs(all_pdf_documents)

In [6]:
#Embedding Manager

from sentence_transformers import SentenceTransformer
class EmbeddingManager:
    def __init__(self, model_name = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = SentenceTransformer(self.model_name)

    def generate_embeddings(self,text):
        embeddings = self.model.encode(text, show_progress_bar = True)
        print("embedding shape:",embeddings.shape)
        return embeddings

In [7]:
embedding_manager = EmbeddingManager()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2472.77it/s]


In [8]:
import chromadb
import uuid

In [9]:
class VectorStoreManager:
    def __init__(self,persist_directory = "data/vector_store", collection_name = "pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        
        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok = True)

        #create a client
        self.client = chromadb.PersistentClient(path = self.persist_directory)

        #create the collection
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description":"vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents)!= len(embeddings):
            raise ValueError("num of documents docs not match num of embeddings")

        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids = ids,
                metadatas = all_metadata,
                documents = documents_content,
                embeddings = embeddings_list
            )

        print("total document added in vector store = ", len(documents_content))
        print("docs in collection:", self.collection.count())

In [10]:
vector_store = VectorStoreManager() 

initialized the vector store with collection: pdf_documents
docs in collection: 1490


In [11]:
texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks,embedding)

Batches: 100%|██████████| 24/24 [01:12<00:00,  3.02s/it]


embedding shape: (745, 384)
total document added in vector store =  745
docs in collection: 2235


In [12]:
#Retriever pipeline
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k = 5, score_threshold = 0.0):
        #query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        #semantic search
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        #cosine simliarity
        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1-distance 

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id":doc_id,
                        "document":document,
                        "metadata":metadata,
                        "distance":distance,
                        "similarity_score":similarity_score,
                        "rank": i+1
                    })

        else:
            print("no documents found")

        return retrieved_docs
        

In [13]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)
rag_retriever.retrieve("What is the Heritage")

Batches: 100%|██████████| 1/1 [00:00<00:00, 14.98it/s]

embedding shape: (1, 384)


[{'id': 'doc170f53ed-cd54-4848-8ca1-257ae3cacdf2',
  'document': "natural assets  that are inherited from the \npast and passed down to future generations. \nIt encompasses a society's collective \nmemory, practices, traditions, and material \nculture. In the case of India, its heritage is not \nmerely a relic of history but a dynamic and \nliving force that shapes our identity, informs \nour present, and guides our future.  \nA deep understanding of this heritage is \nessential, as it provides a sense of belonging, \noffers profound lessons, and continues to",
  'metadata': {'moddate': '2025-06-21T18:09:48+05:30',
   'creator': 'Microsoft® Word 2021',
   'page': 5,
   'producer': 'Microsoft® Word 2021',
   'doc_index': 46,
   'total_pages': 117,
   'creationdate': '2025-06-21T18:09:14+05:30',
   'content_length': 489,
   'author': 'VISION',
   'page_label': '6',
   'source': 'data/pdfs\\history.pdf'},
  'distance': 0.6638541221618652,
  'similarity_score': 0.33614587783813477,
  'rank

In [ ]:
API_KEY_GROQ = "GROQ API KEY"

from langchain_groq import ChatGroq
llm = ChatGroq(
    groq_api_key = API_KEY_GROQ,
    model = "qwen/qwen3.6-27b",
    temperature = 0.1,
    max_tokens = 1024
)

def generate_output(query, retriever, llm, top_k = 3):
    results = retriever.retrieve(query,top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    prompt = f"""use given contxt to generate the answer for thr query
        context:{context}
        Query:{query}"""
    response = llm.invoke([prompt.format(context = context, query = query)])
    return response.content

In [30]:
answer = generate_output("what is AI?",rag_retriever,llm)
print(answer)

Batches: 100%|██████████| 1/1 [00:00<00:00, 12.07it/s]


embedding shape: (1, 384)

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Context:** (Empty/Not provided)
   - **Query:** "what is AI?"
   - **Instruction:** "use given contxt to generate the answer for thr query"

2.  **Identify Key Constraints:**
   - The user explicitly says "use given contxt" (typo: context).
   - However, the context provided is empty.
   - I need to handle this gracefully while still answering the query.

3.  **Determine Response Strategy:**
   - Acknowledge the missing context.
   - Provide a clear, concise, and accurate answer to "what is AI?" based on general knowledge.
   - Note that since no context was provided, I'm using standard knowledge.
   - Keep it structured and helpful.

4.  **Draft Response (Mental Refinement):**
   Since you didn't provide a specific context, I’ll answer based on general knowledge:
   Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think, learn,